# VisionPlate YOLOv8s Local Training Notebook

Use this notebook to train the number plate detector locally on RTX GPU, save graphs for the report, and resume training if interrupted.

## 1. Install Requirements
Run once. Restart the kernel if packages were newly installed.

In [1]:
!pip install ultralytics roboflow opencv-python matplotlib pandas seaborn

Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account al

## 2. Check GPU

In [3]:
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.14.0
Uninstalling torch-2.14.0:
  Successfully uninstalled torch-2.14.0
Found existing installation: torchvision 0.29.0
Uninstalling torchvision-0.29.0:
  Successfully uninstalled torchvision-0.29.0


You can safely remove it manually.


In [4]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 1.2 MB/s eta 0:34:06
     ---------------------------------------- 0.0/2.4 GB 1.0 MB/s eta 0:38:56
     ---------------------------------------- 0.0/2.4 GB 1.2 MB/s eta 0:34:06
     ---------------------------------------- 0.0/2.4 GB 1.1 MB/s eta 0:36:30
     ---------------------------------------- 0.0/2.4 GB 1.1 MB/s eta 0:36:59
     ---------------------------------------- 0.0/2.4 GB 1.1 MB/s eta 0:36:59
     ---------------------------------------- 0.0/2.4 GB 1.1 MB/s eta 0:36:53
     ---------------------------------------- 0.0/2.4 GB 1.1 MB/s eta 0:38:34
     ---------------------------

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 3. Download Dataset From Roboflow
Replace the values with your Roboflow dataset snippet. Choose YOLOv8 format.

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="AJ4uKoH91ufHWzsNIvoG")
project = rf.workspace("abduls-workspace-qwhgu").project("indian-license-plate-merged")
version = project.version(2)
dataset = version.download("yolov8")
                
DATA_YAML = f'{dataset.location}/data.yaml'
print(DATA_YAML)

Defaulting to user installation because normal site-packages is not writeable
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Indian-License-Plate-Merged-2 in yolov8:: 100%|█| 19129/19129 [00:27<00:00, 701.49it/


WARNING Ultralytics settings updated to the latest schema. Existing values were preserved where possible. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\vamsi\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
C:\Users\vamsi\Indian-License-Plate-Merged-2/data.yaml


## 4. Train YOLOv8s
Start with `batch=4` on RTX 3050. If VRAM allows, increase to `batch=8`.

In [3]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=4,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    degrees=8,
    translate=0.08,
    scale=0.4,
    shear=2,
    perspective=0.0005,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.08,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project='../runs',
    name='yolov8s_indian_plate'
)

Ultralytics 8.4.138  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\vamsi\Indian-License-Plate-Merged-2/data.yaml, degrees=8, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.08, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scal


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\ProgramData\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



RuntimeError: Dataset 'C://Users/vamsi/Indian-License-Plate-Merged-2/data.yaml' error  initialization failed

## 5. Resume Training If Interrupted
Run this cell only if training stopped before completion.

In [ ]:
from ultralytics import YOLO

last_weights = '../runs/yolov8s_indian_plate/weights/last.pt'
model = YOLO(last_weights)
model.train(resume=True)

## 6. Validate Best Model

In [ ]:
from ultralytics import YOLO

best_model = YOLO('../runs/yolov8s_indian_plate/weights/best.pt')
metrics = best_model.val(data=DATA_YAML, imgsz=640)
print('mAP@0.5:', metrics.box.map50)
print('mAP@0.5:0.95:', metrics.box.map)

## 7. Run Sample Predictions
These generated images are useful as screenshots for your report.

In [ ]:
best_model.predict(
    source=f'{dataset.location}/test/images',
    imgsz=640,
    conf=0.25,
    save=True,
    project='../runs',
    name='sample_predictions'
)

## 8. Display Training Graphs

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path('../runs/yolov8s_indian_plate')
for image_name in ['results.png', 'confusion_matrix.png', 'P_curve.png', 'R_curve.png', 'F1_curve.png', 'PR_curve.png']:
    image_path = run_dir / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path)))

## 9. Copy Best Model To Backend

In [ ]:
from pathlib import Path
import shutil

source = Path('../runs/yolov8s_indian_plate/weights/best.pt')
destination = Path('../backend/models/yolo/best.pt')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)
print('Copied to:', destination.resolve())